# Garim Stub Worker (Colab)

백엔드 Worker API를 폴링하여 분석 작업을 처리하는 stub 워커입니다.

## 사용법
1. **Cell 1 (설정)** — `BACKEND_URL`, `WORKER_SECRET` 입력
2. **Cell 2 (설치)** — 실행
3. **Cell 3 (함수 정의)** — 실행
4. **Cell 4 (ngrok 속도 측정)** — 파일 전송 속도 확인 (선택)
5. **Cell 5 (워커 시작)** — 실행 후 로그 확인

### ngrok으로 로컬 백엔드 외부 공개 방법
```bash
# 로컬 PC 터미널에서 실행
ngrok http 8000
# 출력된 https://xxxx.ngrok-free.app 를 BACKEND_URL에 입력
```

In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────
BACKEND_URL     = "https://xxxx.ngrok-free.app"        # ngrok URL 또는 서버 주소
WORKER_SECRET   = "change_me_to_a_long_random_secret"  # .env 의 WORKER_SECRET 값
WORKER_ID       = "colab-stub-worker-01"                # 이 워커 식별자 (자유롭게 변경)
POLL_INTERVAL   = 5    # 폴링 간격 (초)
HEARTBEAT_EVERY = 10   # heartbeat 전송 간격 (초)

In [ ]:
# ── 패키지 설치 (Colab 기본 제공 패키지만 사용하므로 생략 가능) ──────
import requests
print(f"requests {requests.__version__} 준비 완료")

In [ ]:
import time
import threading
from datetime import datetime

HEADERS = {
    "Authorization": f"Bearer {WORKER_SECRET}",
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true",
}

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# ── API 호출 헬퍼 ────────────────────────────────────────────────────
def api_get(path):
    return requests.get(f"{BACKEND_URL}{path}", headers=HEADERS, timeout=10)

def api_post(path, body=None):
    return requests.post(f"{BACKEND_URL}{path}", json=body or {}, headers=HEADERS, timeout=10)

def api_put(path, body=None):
    return requests.put(f"{BACKEND_URL}{path}", json=body or {}, headers=HEADERS, timeout=10)

# ── Worker API 래퍼 ──────────────────────────────────────────────────
def poll_next_job():
    r = api_get("/worker/jobs/next")
    if r.status_code == 200:
        return r.json().get("job")
    log(f"poll_next_job 오류: {r.status_code} {r.text}")
    return None

def accept_job(job_id):
    r = api_post(f"/worker/jobs/{job_id}/accept", {"worker_id": WORKER_ID})
    return r.status_code == 200

def report_progress(job_id, stage_name, stage_progress, total_progress, message=""):
    api_put(f"/worker/jobs/{job_id}/progress", {
        "worker_id": WORKER_ID,
        "stage_name": stage_name,
        "stage_progress": stage_progress,
        "total_progress": total_progress,
        "message": message,
    })

def complete_job(job_id, detection_count=0):
    r = api_post(f"/worker/jobs/{job_id}/complete", {
        "worker_id": WORKER_ID,
        "detection_count": detection_count,
    })
    return r.status_code == 200

def fail_job(job_id, error_code, error_message):
    api_post(f"/worker/jobs/{job_id}/fail", {
        "worker_id": WORKER_ID,
        "error_code": error_code,
        "error_message": error_message,
    })

def send_heartbeat(job_id, stage_name, progress):
    api_post("/worker/heartbeat", {
        "job_id": job_id,
        "worker_id": WORKER_ID,
        "worker_type": "colab",
        "current_stage": stage_name,
        "progress": progress,
        "message": f"{stage_name} 처리 중 ({progress}%)",
    })

# ── Stub 분석 처리 ───────────────────────────────────────────────────
STUB_STAGES = [
    ("preparing",      2,  10),
    ("face_detection", 4,  40),
    ("text_detection", 4,  70),
    ("audio_analysis", 3,  90),
    ("finalizing",     2, 100),
]

def run_stub_analysis(job):
    job_id   = job["job_id"]
    filename = job.get("original_filename", "(알 수 없음)")
    log(f"[{job_id[:8]}] 처리 시작: {filename}")

    prev_total = 0
    last_hb    = time.time()

    for stage_name, stage_sec, end_total in STUB_STAGES:
        steps       = max(stage_sec, 1)
        start_total = prev_total

        for step in range(steps + 1):
            stage_pct = int(step / steps * 100)
            total_pct = int(start_total + (end_total - start_total) * step / steps)
            report_progress(job_id, stage_name, stage_pct, total_pct, f"{stage_name} {stage_pct}% 완료")

            now = time.time()
            if now - last_hb >= HEARTBEAT_EVERY:
                send_heartbeat(job_id, stage_name, total_pct)
                last_hb = now

            log(f"  [{job_id[:8]}] {stage_name}: {stage_pct}% (전체 {total_pct}%)")
            if step < steps:
                time.sleep(1)

        prev_total = end_total

    stub_detection_count = 3
    ok = complete_job(job_id, detection_count=stub_detection_count)
    if ok:
        log(f"[{job_id[:8]}] 완료 — 탐지 {stub_detection_count}건 (stub)")
    else:
        log(f"[{job_id[:8]}] complete_job 호출 실패")

log("함수 정의 완료")

In [ ]:
# ── Cell 4 : ngrok 경유 파일 전송 속도 측정 ──────────────────────────
# 대기 중인 잡의 파일을 자동으로 가져와 다운로드 속도를 측정합니다.
# 잡 상태는 변경하지 않으므로 워커 루프 실행 전에 안전하게 사용할 수 있습니다.

job = poll_next_job()
if job is None:
    log("대기 중인 잡이 없습니다. 먼저 파일을 업로드하고 분석 잡을 생성하세요.")
else:
    upload_id = job["upload_id"]
    filename  = job.get("original_filename", "(알 수 없음)")
    file_size = job.get("file_size", 0)
    log(f"파일: {filename} / {file_size/1024/1024:.2f} MB")
    log("다운로드 시작...")

    download_headers = {k: v for k, v in HEADERS.items() if k != "Content-Type"}
    start = time.time()
    r = requests.get(
        f"{BACKEND_URL}/worker/files/{upload_id}",
        headers=download_headers,
        stream=True,
        timeout=300,
    )

    if r.status_code != 200:
        log(f"오류: {r.status_code} {r.text}")
    else:
        total_bytes = 0
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            total_bytes += len(chunk)
            elapsed = time.time() - start
            speed = total_bytes / elapsed / 1024 / 1024 if elapsed > 0 else 0
            print(f"\r  수신: {total_bytes/1024/1024:.1f} MB | {speed:.1f} MB/s", end="")
        elapsed = time.time() - start
        speed = total_bytes / elapsed / 1024 / 1024 if elapsed > 0 else 0
        print()
        log(f"완료: {total_bytes/1024/1024:.2f} MB / {elapsed:.1f}초 / 평균 {speed:.2f} MB/s")

In [ ]:
# ── Cell 5 : 워커 메인 루프 ───────────────────────────────────────────
# 중단하려면 Colab 의 [■ 중단] 버튼 또는 런타임 메뉴 > 실행 중단 클릭

log(f"워커 시작 — backend: {BACKEND_URL}")
log(f"워커 ID: {WORKER_ID} / 폴링 간격: {POLL_INTERVAL}s")

while True:
    try:
        job = poll_next_job()

        if job is None:
            log("대기 중인 작업 없음. 재폴링...")
            time.sleep(POLL_INTERVAL)
            continue

        job_id = job["job_id"]
        log(f"작업 발견: {job_id[:8]} (upload: {job['upload_id'][:8]})")

        if not accept_job(job_id):
            log(f"[{job_id[:8]}] accept 실패 — 다른 워커가 선점했을 가능성")
            time.sleep(1)
            continue

        log(f"[{job_id[:8]}] 수락 완료 → 분석 시작")

        try:
            run_stub_analysis(job)
        except Exception as e:
            log(f"[{job_id[:8]}] 처리 중 오류: {e}")
            fail_job(job_id, "STUB_ERROR", str(e))

    except KeyboardInterrupt:
        log("워커 중단됨")
        break
    except Exception as e:
        log(f"루프 오류: {e}")
        time.sleep(POLL_INTERVAL)